# 02 · News agent
Documentation: `docs/agents/news_agent.md`.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_rows', 80); pd.set_option('display.width', 200)
D = ROOT / 'data'
def ok(p):
    if not Path(p).exists(): print('File not available yet:', p); return False
    return True

In [ ]:
if ok(D/'news/news.parquet'):
    n = pd.read_parquet(D/'news/news.parquet')
    n['day'] = pd.to_datetime(n.fetched_at, utc=True).dt.date
    print('News items:', len(n), '· days:', n.day.nunique())
    display(n.groupby(['day','theme']).size().unstack(fill_value=0))

## Sentiment and impact by theme

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
n.boxplot(column='sentiment', by='theme', ax=ax[0], rot=45); n.groupby('theme').impact.mean().plot(kind='bar', ax=ax[1], title='Average impact')
plt.suptitle(''); plt.tight_layout(); plt.show()

## Sources and tickers

In [ ]:
display(n.source_name.value_counts().head(20))
tk = n.explode('tickers').tickers.value_counts().head(25); display(tk.rename('mentions'))
print('Tickers proposed outside the universe:', n.explode('unknown_tickers').unknown_tickers.dropna().value_counts().head(20).to_dict())

## Signals (impact-weighted sentiment, half-life 3 days)

In [ ]:
s = pd.read_parquet(D/'processed/news_signals.parquet')
for k in ['ticker','asset_class','region']:
    print(k); display(s[s.kind==k].sort_values('score')[['key','score','weight','n_news']].round(3))

## Latest high-impact news

In [ ]:
n.sort_values(['fetched_at','impact'], ascending=False).query('impact >= 4')[['day','theme','title','tickers','sentiment','impact','horizon']].head(25)

## Did sentiment anticipate prices? (meaningful only after weeks of history)

In [ ]:
px = pd.read_parquet(D/'processed/prices.parquet')
e = n.explode('tickers').dropna(subset=['tickers'])
e['date'] = pd.to_datetime(e.fetched_at, utc=True).dt.tz_localize(None).dt.normalize()
def fwd(row, days=5):
    s = px[row.tickers].dropna(); s = s.loc[row.date:]
    return s.iloc[min(days, len(s)-1)] / s.iloc[0] - 1 if len(s) > 1 else np.nan
e['ret_5d'] = e.apply(fwd, axis=1)
print('Correlation sentiment / 5-day return:', round(e[['sentiment','ret_5d']].corr().iloc[0,1], 3), '· observations:', e.ret_5d.notna().sum())